In [1]:
from pathlib import Path
import re
import pandas as pd

In [2]:
BASE_DIR = Path("/home/alex/Desktop/HSE/Молекулярная эволюция/Дз 5")
RESULTS_DIR = BASE_DIR / "results"

result_files = sorted(RESULTS_DIR.glob("PF*.txt"))

print(f"Найдено файлов: {len(result_files)}")
for file in result_files:
    print(file.name)

Найдено файлов: 13
PF00015.alignment.txt
PF00069.alignment.txt
PF00211.alignment.txt
PF00512.alignment.txt
PF00563.alignment.txt
PF00990.alignment.txt
PF01295.alignment.txt
PF01928.alignment.txt
PF01966.alignment.txt
PF02518.alignment.txt
PF07536.alignment.txt
PF07568.alignment.txt
PF07730.alignment.txt


In [3]:
def parse_hmmer(file_path):
    text = Path(file_path).read_text(encoding="utf-8")

    if "Scores for complete sequences" not in text:
        raise ValueError(f"Не найдена таблица результатов: {file_path}")

    section = text.split("Scores for complete sequences", 1)[1]
    section = section.split("Domain annotation for each sequence", 1)[0]
    section = section.split("------ inclusion threshold ------", 1)[0]

    proteins = set()

    for line in section.splitlines():
        columns = line.split()

        if len(columns) < 9:
            continue

        try:
            float(columns[0])   # E-value
            float(columns[3])   # E-value лучшего домена
            int(columns[7])     # Количество доменных совпадений
        except ValueError:
            continue

        protein_id = columns[8]
        proteins.add(protein_id)

    return proteins

In [4]:
mcp_file = RESULTS_DIR / "PF00015.alignment.txt"

mcp_proteins = parse_hmmer(mcp_file)

print(f"Найдено MCP-белков: {len(mcp_proteins)}")
print(sorted(mcp_proteins))

Найдено MCP-белков: 21
['WP_011550321.1', 'WP_011550999.1', 'WP_011551531.1', 'WP_011552751.1', 'WP_011553486.1', 'WP_011553686.1', 'WP_011553767.1', 'WP_011554144.1', 'WP_011555119.1', 'WP_011555120.1', 'WP_011555976.1', 'WP_011556387.1', 'WP_011556858.1', 'WP_011556871.1', 'WP_020477981.1', 'WP_020478209.1', 'WP_026113956.1', 'WP_026114142.1', 'WP_044272907.1', 'WP_171452280.1', 'WP_225909887.1']


In [5]:
hits = {}

for file in result_files:
    match = re.search(r"PF\d{5}", file.name)

    if match is None:
        continue

    pfam_id = match.group()

    if pfam_id in hits:
        raise ValueError(f"Повторяется домен {pfam_id}")

    hits[pfam_id] = parse_hmmer(file)

print(f"Обработано доменов: {len(hits)}")

Обработано доменов: 13


In [6]:
required_domains = {
    "PF00512", "PF07568", "PF07730", "PF07536",
    "PF02518", "PF00015", "PF00069", "PF00990",
    "PF01295", "PF01928", "PF00211", "PF00563",
    "PF01966"
}

missing = required_domains - hits.keys()

if missing:
    raise ValueError(f"Отсутствуют результаты для: {sorted(missing)}")

In [7]:
required_domains = {
    "PF00512", "PF07568", "PF07730", "PF07536",
    "PF02518", "PF00015", "PF00069", "PF00990",
    "PF01295", "PF01928", "PF00211", "PF00563",
    "PF01966"
}

missing = required_domains - hits.keys()

if missing:
    raise ValueError(f"Отсутствуют результаты для: {sorted(missing)}")

In [8]:
# Гистидинкиназы: фосфоакцептор + ATPase
phosphoacceptors = (
    hits["PF00512"]
    | hits["PF07568"]
    | hits["PF07730"]
    | hits["PF07536"]
)

histidine_kinases = phosphoacceptors & hits["PF02518"]

# Остальные категории
categories = {
    "Histidine kinases": histidine_kinases,

    "MCP": hits["PF00015"],

    "Ser/Thr/Tyr kinases": hits["PF00069"],

    "Diguanylate cyclases": hits["PF00990"],

    "Adenylate cyclases": (
        hits["PF01295"]
        | hits["PF01928"]
        | hits["PF00211"]
    ),

    "c-di-GMP phosphodiesterases": (
        hits["PF00563"]
        | hits["PF01966"]
    )
}

In [9]:
summary_df = pd.DataFrame([
    {
        "Тип сигнальных белков": category,
        "Количество": len(proteins)
    }
    for category, proteins in categories.items()
])

display(summary_df)

,Тип сигнальных белков,Количество
0,Histidine kinases,121
1,MCP,21
2,Ser/Thr/Tyr kinases,101
3,Diguanylate cyclases,18
4,Adenylate cyclases,13
5,c-di-GMP phosphodiesterases,16


In [10]:
all_signaling_proteins = set().union(*categories.values())

total_signaling = len(all_signaling_proteins)

print(f"Всего уникальных сигнальных белков: {total_signaling}")

Всего уникальных сигнальных белков: 285


In [11]:
summary_df.loc[len(summary_df)] = {
    "Тип сигнальных белков": "TOTAL (unique)",
    "Количество": total_signaling
}

display(summary_df)

,Тип сигнальных белков,Количество
0,Histidine kinases,121
1,MCP,21
2,Ser/Thr/Tyr kinases,101
3,Diguanylate cyclases,18
4,Adenylate cyclases,13
5,c-di-GMP phosphodiesterases,16
6,TOTAL (unique),285


In [12]:
overlap_df = pd.DataFrame(
    {
        category_1: {
            category_2: len(proteins_1 & proteins_2)
            for category_2, proteins_2 in categories.items()
        }
        for category_1, proteins_1 in categories.items()
    }
)

display(overlap_df)

,Histidine kinases,MCP,Ser/Thr/Tyr kinases,Diguanylate cyclases,Adenylate cyclases,c-di-GMP phosphodiesterases
Histidine kinases,121,0,1,2,0,0
MCP,0,21,0,0,0,0
Ser/Thr/Tyr kinases,1,0,101,0,2,0
Diguanylate cyclases,2,0,0,18,0,0
Adenylate cyclases,0,0,2,0,13,0
c-di-GMP phosphodiesterases,0,0,0,0,0,16


In [16]:
summary_df.to_csv(
    BASE_DIR / "signaling_proteins_summary.csv",
    index=False,
    sep="\t"
)

overlap_df.to_csv(
    BASE_DIR / "category_overlaps.csv",
    sep="\t"
)